# 05 — Calibration, fidelity probe, gating thresholds

1. Collect uncertainty scores + correctness labels on the val split per tool.
2. Fit temperature scaling → ECE < 0.05 target.
3. Run fidelity probe on the held-out probe split → pick τ_k for ≥92% match.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from dyna_grpo.config import MODEL, PATHS, DYNA
from dyna_grpo.utils import read_jsonl, save_metrics
from dyna_grpo.predictor import (ToolPredictor, fit_temperature, expected_calibration_error,
                                  fidelity_score, pick_threshold)

In [ ]:
def load_predictor(tool):
    out_dir = Path(PATHS['ckpts']) / f'predictor_{tool}'
    tok = AutoTokenizer.from_pretrained(out_dir, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        MODEL.predictor_base, torch_dtype=torch.bfloat16, device_map='cuda:0', trust_remote_code=True)
    base = PeftModel.from_pretrained(base, out_dir)
    pred = ToolPredictor(base, base.config.hidden_size).to('cuda:0').to(torch.bfloat16)
    aux = torch.load(out_dir / 'aux.pt', map_location='cuda:0')
    pred.unc_head.load_state_dict(aux['unc_head'])
    pred.temperature.data = aux['temperature'].to(pred.temperature.device).to(pred.temperature.dtype)
    return pred, tok

In [ ]:
results = {}
for tool in ('calc', 'code', 'search'):
    print(f'=== {tool} ===')
    pred, tok = load_predictor(tool)
    val = read_jsonl(Path(PATHS['traces']) / f'{tool}_val.jsonl')[:300]
    probe = read_jsonl(Path(PATHS['traces']) / f'{tool}_probe.jsonl')[:500]

    # 1. Predict + score on val
    logits, labels, fidelities = [], [], []
    for r in val:
        out = pred.predict(tok, tool, r['args'])
        f = fidelity_score(out.text, r.get('output') or '', tool)
        logits.append(out.raw_score)
        labels.append(1 if f >= 0.99 else 0)
        fidelities.append(f)
    L = torch.tensor(logits)
    Y = torch.tensor(labels)

    # 2. Fit temperature
    T = fit_temperature(pred, L.to('cuda:0'), Y.to('cuda:0'))
    probs = torch.sigmoid(L / T)
    ece = expected_calibration_error(probs, Y)
    print(f'  Temperature={T:.3f}, ECE={ece:.4f}')

    # 3. Fidelity probe → threshold
    p_logits, p_unc, p_fid = [], [], []
    for r in probe:
        out = pred.predict(tok, tool, r['args'])
        p_unc.append(out.uncertainty)
        p_fid.append(fidelity_score(out.text, r.get('output') or '', tool))
    tau = pick_threshold(p_unc, p_fid, target=DYNA.fidelity_target)
    avg_fid = sum(p_fid) / len(p_fid) if p_fid else 0
    print(f'  Fidelity overall={avg_fid:.3f}, τ_{tool}={tau:.3f}')
    results[tool] = {'temperature': T, 'ece': ece, 'threshold': tau,
                      'avg_fidelity': avg_fid, 'n_probe': len(probe)}
    # Persist updated temperature
    out_dir = Path(PATHS['ckpts']) / f'predictor_{tool}'
    torch.save({'unc_head': pred.unc_head.state_dict(),
                'temperature': pred.temperature.detach().cpu()}, out_dir / 'aux.pt')
    del pred; torch.cuda.empty_cache()

save_metrics(Path(PATHS['logs']) / 'calibration.json', results)
print(results)